# A4: Criticality & Vulnerability Analysis

**Criticality** (Jafino et al. 2020, Metric 10 — weighted betweenness centrality):
- How often a bridge/segment appears on shortest paths × how many tonnes of goods it carries daily
- `criticality_score = betweenness_centrality × daily_tonnes`

**Vulnerability** (Berdica 2002, Jafino et al. 2020, Metric 11 — disaster exposure overlay):
- Bridge structural condition × flood risk exposure
- `vulnerability_score = condition_weight × flood_risk` (flood_risk = placeholder until data provided)

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np

# ── Truck tonnage assumptions (standard Bangladesh freight vehicles) ──
# Source: reasonable approximations for reporting purposes
HEAVY_TRUCK_TONNES  = 25  # tonnes per heavy truck trip
MEDIUM_TRUCK_TONNES = 10  # tonnes per medium truck trip
SMALL_TRUCK_TONNES  =  3  # tonnes per small truck trip

# Bridge condition → vulnerability weight
CONDITION_WEIGHT = {'A': 1, 'B': 2, 'C': 3, 'D': 4}

DATA_DIR = '../data/data_processed/'
print('Constants loaded')

## Step 1: Build NetworkX graph from road network CSV

In [ ]:
df = pd.read_csv(DATA_DIR + 'A3_network_roads.csv')
print(f'Loaded {len(df)} rows, roads: {df["road"].unique().tolist()}')
df.head(3)

In [ ]:
# Build graph — same logic as model.py but without Mesa
G = nx.Graph()

for road in df['road'].unique():
    road_df = df[df['road'] == road]
    prev_id = None
    for _, row in road_df.iterrows():
        node_id = int(row['id'])
        if not G.has_node(node_id):
            G.add_node(node_id,
                       model_type=row['model_type'].strip(),
                       name=str(row['name']),
                       road=row['road'],
                       lat=row['lat'],
                       lon=row['lon'],
                       length=row['length'],
                       condition=row.get('condition', None))
        if prev_id is not None:
            G.add_edge(prev_id, node_id, weight=row['length'], length=row['length'])
        prev_id = node_id

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
bridges_in_graph = [n for n, d in G.nodes(data=True) if d.get('model_type') == 'bridge']
print(f'Bridge nodes: {len(bridges_in_graph)}')

## Step 2: Betweenness centrality
How often does each node (bridge/segment) appear on the shortest path between all pairs of nodes?
Higher score = more critical to the network structure.

In [ ]:
print('Computing betweenness centrality (this may take ~1 min for large graphs)...')
betweenness = nx.betweenness_centrality(G, weight='length', normalized=True)
print(f'Done. Max betweenness: {max(betweenness.values()):.4f}')

## Step 3: Daily tonnes per road from AADT data

In [ ]:
df_aadt = pd.read_csv(DATA_DIR + 'traffic_aadt.csv')
print(df_aadt.columns.tolist())
df_aadt.head(3)

In [ ]:
# Compute daily tonnes per road segment, then average per road
df_aadt['daily_tonnes'] = (
    df_aadt['heavy_truck']  * HEAVY_TRUCK_TONNES +
    df_aadt['medium_truck'] * MEDIUM_TRUCK_TONNES +
    df_aadt['small_truck']  * SMALL_TRUCK_TONNES
)

road_daily_tonnes = df_aadt.groupby('road')['daily_tonnes'].mean().to_dict()
print('Daily tonnes per road (average):')
for road, tonnes in sorted(road_daily_tonnes.items()):
    print(f'  {road}: {tonnes:,.0f} tonnes/day')

## Step 4: Build bridge analysis DataFrame

In [ ]:
bridge_rows = []
for node_id, data in G.nodes(data=True):
    if data.get('model_type') == 'bridge':
        road = data['road']
        bc   = betweenness.get(node_id, 0.0)
        dt   = road_daily_tonnes.get(road, 0.0)
        cond = str(data.get('condition', '')).strip().upper()
        cw   = CONDITION_WEIGHT.get(cond, None)

        bridge_rows.append({
            'agent_id'          : node_id,
            'name'              : data['name'],
            'road'              : road,
            'lat'               : data['lat'],
            'lon'               : data['lon'],
            'length_m'          : data['length'],
            'condition'         : cond,
            'betweenness'       : bc,
            'daily_tonnes'      : dt,
            'criticality_score' : bc * dt,
            'condition_weight'  : cw,
            # ── Flood risk placeholder ──────────────────────────────────
            # TODO: fill flood_risk once natural disaster data is provided
            # Replace None with a 0-1 normalised flood depth value per bridge
            'flood_risk'        : None,
            # vulnerability_score = condition_weight * flood_risk (when available)
            # For now: vulnerability = condition_weight only
            'vulnerability_score': cw,
        })

df_bridges = pd.DataFrame(bridge_rows)
print(f'Bridges analysed: {len(df_bridges)}')
df_bridges.head()

## Step 5: Save results

In [ ]:
out_path = DATA_DIR + 'bridge_analysis.csv'
df_bridges.to_csv(out_path, index=False)
print(f'Saved to {out_path}')

## Step 6: Top-10 results

In [ ]:
print('=== TOP-10 MOST CRITICAL BRIDGES (betweenness × daily tonnes) ===')
top_critical = df_bridges.nlargest(10, 'criticality_score')[['name','road','condition','betweenness','daily_tonnes','criticality_score']]
print(top_critical.to_string(index=False))

In [ ]:
print('=== TOP-10 MOST VULNERABLE BRIDGES (condition weight — flood data pending) ===')
top_vuln = df_bridges[df_bridges['condition_weight'].notna()].nlargest(10, 'vulnerability_score')[['name','road','condition','condition_weight','vulnerability_score']]
print(top_vuln.to_string(index=False))

## Step 7: Update vulnerability with flood data (fill this in later)

When you have the natural disaster / flood data:
1. Load flood depth per bridge (see `preprocessing/flood_risk.py`)
2. Merge into `df_bridges` on `agent_id`
3. Replace `flood_risk` column with actual values (normalised 0–1)
4. Recompute: `vulnerability_score = condition_weight × flood_risk`
5. Re-save `bridge_analysis.csv`

In [ ]:
# TODO: plug in flood data here
# df_flood = pd.read_csv(DATA_DIR + 'flood_risk.csv')  # columns: agent_id, flood_risk
# df_bridges = df_bridges.merge(df_flood, on='agent_id', how='left', suffixes=('_old',''))
# df_bridges['vulnerability_score'] = df_bridges['condition_weight'] * df_bridges['flood_risk']
# df_bridges.to_csv(out_path, index=False)
print('Flood data stub — fill in when data is available')